<a href="https://colab.research.google.com/github/chiraswykj/GENAI_TUTORIALS/blob/main/GENAI5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pymupdf sentence-transformers faiss-cpu transformers accelerate

from google.colab import files


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.9 MB/s eta 0:00:00


In [ ]:
uploaded = files.upload()

pdf_path = next(iter(uploaded))

if not pdf_path.lower().endswith(".pdf"):
    raise ValueError("Please upload a PDF file only.")

print("PDF uploaded successfully!")
print("File:", pdf_path)

Saving sample_rag_document.pdf to sample_rag_document.pdf
PDF uploaded successfully!
File: sample_rag_document.pdf


In [ ]:
import fitz

def extract_pdf(pdf_path):

    document = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(document, start=1):

        text = page.get_text("text")

        if text.strip():

            pages.append({
                "page": page_number,
                "text": text
            })

    document.close()

    return pages


In [ ]:
pages = extract_pdf(pdf_path)

print("Total pages:", len(pages))

if len(pages) == 0:
    print("No text found. Your PDF may be scanned and require OCR.")
else:
    print("\nFirst page preview:\n")
    print(pages[0]["text"][:1000])

Total pages: 1

First page preview:

Introduction to Artificial Intelligence
Artificial Intelligence (AI) is the field of computer science concerned with creating systems that can
perform tasks that normally require human intelligence.
Machine Learning
Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from
data and make predictions or decisions without being explicitly programmed for every task.
Deep Learning
Deep Learning is a branch of machine learning that uses neural networks with multiple layers. It is
widely used for image recognition, speech processing, natural language processing, and
recommendation systems.
Natural Language Processing
Natural Language Processing (NLP) allows computers to understand, process, and generate
human language. Applications include chatbots, translation systems, sentiment analysis, and
document question answering.
Retrieval-Augmented Generation
Retrieval-Augmented Generation (RAG) combines information retr

In [ ]:
def create_chunks(pages, chunk_size=1000, overlap=200):

    chunks = []

    for page in pages:

        text = page["text"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end].strip()

            if chunk_text:

                chunks.append({
                    "text": chunk_text,
                    "page": page["page"]
                })

            start += chunk_size - overlap

    return chunks
chunks = create_chunks(pages)

print("Total chunks:", len(chunks))


Total chunks: 3


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (3, 384)


In [ ]:
import faiss
import numpy as np

embeddings = embeddings.astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors stored:", index.ntotal)


Vectors stored: 3


In [ ]:
def retrieve(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for distance, idx in zip(distances[0], indices[0]):

        results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "distance": float(distance)
        })

    return results


question = "What is this document about?"

results = retrieve(question)

for i, result in enumerate(results):

    print("=" * 80)
    print("RESULT:", i + 1)
    print("PAGE:", result["page"])
    print("=" * 80)
    print(result["text"])



from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)



def ask_pdf(question, top_k=5):

    results = retrieve(question, top_k)

    context = ""

    for result in results:

        context += (
            f"\n[Page {result['page']}]\n"
            f"{result['text']}\n"
        )

    prompt = f"""
You are a helpful PDF question-answering assistant.

Answer the question using ONLY the information provided
in the context below.

If the answer is not available in the context, say:

"I could not find the answer in the PDF."

Always mention the page number when possible.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    output = generator(
        prompt,
        max_new_tokens=300,
        do_sample=False
    )

    generated_text = output[0]["generated_text"]

    return generated_text, results




question = input("Ask a question about your PDF: ")

answer, sources = ask_pdf(question)

print("\nANSWER")
print("=" * 80)
print(answer)

print("\nSOURCES")
print("=" * 80)

for source in sources:
    print("Page:", source["page"])



while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Chat ended.")
        break

    answer, sources = ask_pdf(question)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCE PAGES:")

    pages_used = sorted(
        set(source["page"] for source in sources)
    )

    print(pages_used)


RESULT: 1
PAGE: 1
a specific collection of documents and can return references to the original
document pages.
Conclusion
PDF-based RAG systems are useful for asking questions about reports, research papers, manuals,
books, company documents, and other collections of unstructured information.
RESULT: 2
PAGE: 1
slation systems, sentiment analysis, and
document question answering.
Retrieval-Augmented Generation
Retrieval-Augmented Generation (RAG) combines information retrieval with large language
models. A RAG system first retrieves relevant information from a knowledge base and then
provides that information to a language model to generate an answer.
PDF-Based RAG
A PDF-based RAG system extracts text from PDF documents, divides the text into smaller chunks,
converts the chunks into numerical embeddings, and stores them in a vector database. When a
user asks a question, the system retrieves the most relevant chunks and provides them to the
language model as context.
Advantages of RAG
RA

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

... (5 outputs left out)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



ANSWER

You are a helpful PDF question-answering assistant.

Answer the question using ONLY the information provided
in the context below.

If the answer is not available in the context, say:

"I could not find the answer in the PDF."

Always mention the page number when possible.

CONTEXT:

[Page 1]
Introduction to Artificial Intelligence
Artificial Intelligence (AI) is the field of computer science concerned with creating systems that can
perform tasks that normally require human intelligence.
Machine Learning
Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from
data and make predictions or decisions without being explicitly programmed for every task.
Deep Learning
Deep Learning is a branch of machine learning that uses neural networks with multiple layers. It is
widely used for image recognition, speech processing, natural language processing, and
recommendation systems.
Natural Language Processing
Natural Language Processing (NLP) al

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:

You are a helpful PDF question-answering assistant.

Answer the question using ONLY the information provided
in the context below.

If the answer is not available in the context, say:

"I could not find the answer in the PDF."

Always mention the page number when possible.

CONTEXT:

[Page 1]
Introduction to Artificial Intelligence
Artificial Intelligence (AI) is the field of computer science concerned with creating systems that can
perform tasks that normally require human intelligence.
Machine Learning
Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from
data and make predictions or decisions without being explicitly programmed for every task.
Deep Learning
Deep Learning is a branch of machine learning that uses neural networks with multiple layers. It is
widely used for image recognition, speech processing, natural language processing, and
recommendation systems.
Natural Language Processing
Natural Language Processing (NLP) a